# TI-748: Causal Impact Analysis — Media Plan Feature (v2)

---

## Glossary & Key Definitions

| Term | Definition |
|---|---|
| **Media Plan** | A beta feature giving advertisers control over network allocation (% of spend per publisher like ABC, CBS, ESPN). MNTN auto-recommends allocations; advertisers can accept or customize. |
| **Recommended** | An advertiser who accepted MNTN's suggested network allocation without modification (`badge_state = 'RECOMMENDED'` on all publishers). |
| **Customized** | An advertiser who modified the allocation (`badge_state = 'USER_MODIFIED'` or `'USER_ADDED'`). |
| **CausalImpact** | Google's Bayesian Structural Time Series method for estimating the causal effect of an intervention on a time series. |
| **Intervention Date** | The date each advertiser first created a recommended media plan (`media_plan.create_time`). |
| **Pre-period** | The time before the intervention — used to train the model on the advertiser's baseline behavior. We use 52 weeks (1 full year) to capture seasonality. |
| **Post-period** | The time after the intervention — the model compares actual performance to what it predicted would have happened without the intervention. |
| **Counterfactual** | The model's prediction of what *would have happened* if the advertiser had NOT adopted media plan. |
| **Relative Effect** | The % difference between actual performance and the counterfactual: `(actual - predicted) / predicted`. Positive = improvement. |
| **p-value** | The probability of seeing this effect (or more extreme) if the intervention had no real impact. p < 0.05 = statistically significant. |
| **Spend-Weighted Average** | An average where each advertiser's effect is weighted by their total post-period spend. This prevents small advertisers (with noisy data) from disproportionately influencing the overall result. A $1M advertiser's -10% counts 10x more than a $100K advertiser's +50%. |
| **Covariates** | External variables the model uses to build a better counterfactual. If IVR always drops during holidays, the model learns that pattern from covariates and doesn't attribute the drop to media plan. |
| **Placebo Test** | A validation test: we pretend the intervention happened at the midpoint of the pre-period. If the model is reliable, it should find NO significant effect (because nothing actually changed). |
| **Winsorize** | Clipping extreme values to the 1st/99th percentile to reduce the influence of data anomalies. |

### Metrics

| Metric | Formula | Better When | What It Measures |
|---|---|---|---|
| **IVR** | verified_visits / impressions | Higher | How often an ad impression leads to a site visit |
| **CVR** | conversions / verified_visits | Higher | How often a visit leads to a conversion |
| **CPA** | spend / conversions | Lower | Cost to acquire one conversion |
| **CPV** | spend / verified_visits | Lower | Cost to generate one site visit |
| **ROAS** | order_value / spend | Higher | Revenue generated per dollar spent |

---

## Executive Summary

**Objective:** Determine if advertisers who adopted MNTN's *recommended* Media Plan network allocation saw improved prospecting performance.

**Two analyses:**
1. **CausalImpact (time-series):** For each adopter, compare their advertiser-level prospecting performance before vs after adopting media plan, controlling for market trends.
2. **Within-advertiser comparison (cross-sectional):** For advertisers with both recommended and non-recommended campaigns, compare performance side-by-side in the same time period.

| Design Element | Choice | Rationale |
|---|---|---|
| Unit of analysis | Advertiser (all prospecting campaigns) | Feature affects portfolio-level network allocation |
| Intervention date | First recommended plan `create_time` | When they actually started using the feature |
| Pre-period | 52 weeks (max available) | Full seasonal cycle — captures Black Friday, holidays, Q1 slowdown |
| Post-period | Variable (4-22 weeks) | Depends on adoption date |
| Covariates | Platform IVR, spend, impressions, holiday flags, VCR, active advertisers | Controls for market trends, seasonality, platform changes |
| Filter | Recommended plans only | Per Kirsa: only care about advertisers who used MNTN's recommended allocation |

---

## 1. Setup & Configuration

In [ ]:
# !pip install google-cloud-bigquery pycausalimpact pandas numpy matplotlib db-dtypes

import warnings
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from causalimpact import CausalImpact
from google.cloud import bigquery

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*frequency information.*")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

BQ_PROJECT = "dw-main-silver"
MIN_WEEKLY_IMPRESSIONS = 1000
MIN_TOTAL_POST_SPEND = 10_000
MIN_PRE_WEEKS = 20
MIN_POST_WEEKS = 4
PRIMARY_METRIC = "ivr"
COVARIATES = ["platform_ivr", "platform_spend", "platform_impressions",
              "holiday", "platform_active_advertisers", "platform_vcr"]

METRIC_DEFS = {
    "ivr":  {"direction": "higher", "label": "Impression-to-Visit Rate"},
    "cvr":  {"direction": "higher", "label": "Conversion Rate"},
    "cpa":  {"direction": "lower",  "label": "Cost per Acquisition"},
    "cpv":  {"direction": "lower",  "label": "Cost per Visit"},
    "roas": {"direction": "higher", "label": "Return on Ad Spend"},
}

HOLIDAY_WEEKS = {
    pd.Timestamp("2024-11-25"): 1, pd.Timestamp("2024-12-23"): 1, pd.Timestamp("2024-12-30"): 1,
    pd.Timestamp("2025-11-24"): 1, pd.Timestamp("2025-12-22"): 1, pd.Timestamp("2025-12-29"): 1,
    pd.Timestamp("2026-02-02"): 1,
}

client = bigquery.Client(project=BQ_PROJECT)
print("Setup complete.")

## 2. Identify Recommended Media Plan Adopters

Only advertisers with `badge_state = 'RECOMMENDED'` on ALL publishers in at least one active plan.

In [ ]:
adopters = client.query("""
WITH plan_status AS (
    SELECT mp.media_plan_id, mp.advertiser_id, mp.campaign_group_id, mp.create_time,
           LOGICAL_AND(mpp.badge_state = 'RECOMMENDED') AS all_recommended
    FROM `dw-main-silver.core.media_plan` mp
    JOIN `dw-main-silver.core.media_plan_publishers` mpp ON mpp.media_plan_id = mp.media_plan_id
    WHERE mp.media_plan_status_id = 3
    GROUP BY 1, 2, 3, 4
)
SELECT ps.advertiser_id, adv.company_name,
    MIN(CASE WHEN ps.all_recommended THEN ps.create_time END) AS first_recommended_plan,
    COUNT(*) AS total_plans,
    COUNTIF(ps.all_recommended) AS recommended_plans,
    COUNTIF(NOT ps.all_recommended) AS customized_plans
FROM plan_status ps
JOIN `dw-main-bronze.integrationprod.advertisers` adv ON adv.advertiser_id = ps.advertiser_id
GROUP BY 1, 2
HAVING COUNTIF(ps.all_recommended) > 0
ORDER BY 3
""").to_dataframe()

adopters["first_recommended_plan"] = pd.to_datetime(adopters["first_recommended_plan"])
adopters["intervention_date"] = adopters["first_recommended_plan"].dt.date

print(f"{len(adopters)} advertisers with recommended media plans")
adopters[["advertiser_id", "company_name", "intervention_date", "recommended_plans", "customized_plans"]]

## 3. Load Weekly Prospecting KPIs

- **Source:** `summarydata.sum_by_campaign_by_day` (2024-01-01 to present — 15+ months of history)
- **Filter:** Prospecting campaigns only (`funnel_level = 1`), non-test, non-deleted
- **Attribution:** All adopters use `industry_standard` (competing views included)
- **Quality:** Weeks with < 1,000 impressions excluded (attribution lag artifacts)

In [ ]:
adopter_list = ",".join(str(x) for x in adopters["advertiser_id"])

weekly_kpis = client.query(f"""
WITH prospecting AS (
    SELECT DISTINCT c.campaign_id, c.advertiser_id, c.campaign_group_id
    FROM `dw-main-bronze.integrationprod.campaigns` c
    WHERE c.funnel_level = 1 AND c.deleted = FALSE AND c.is_test = FALSE
)
SELECT pc.advertiser_id, pc.campaign_group_id,
    DATE_TRUNC(s.day, WEEK(MONDAY)) AS week_start,
    CASE WHEN pc.advertiser_id IN ({adopter_list}) THEN TRUE ELSE FALSE END AS is_adopter,
    SUM(s.impressions) AS impressions,
    SUM(s.media_spend + s.data_spend + s.platform_spend) AS spend,
    SUM(s.clicks + s.views + COALESCE(s.competing_views, 0)) AS vv,
    SUM(s.click_conversions + s.view_conversions + COALESCE(s.competing_view_conversions, 0)) AS conversions,
    SUM(s.click_order_value + s.view_order_value + COALESCE(s.competing_view_order_value, 0)) AS order_value,
    SUM(s.vast_start) AS vast_start, SUM(s.vast_complete) AS vast_complete
FROM `dw-main-silver.summarydata.sum_by_campaign_by_day` s
JOIN prospecting pc ON pc.campaign_id = s.campaign_id
WHERE s.day >= '2024-01-01' AND s.impressions > 0
GROUP BY 1, 2, 3, 4 ORDER BY 1, 3
""").to_dataframe()

weekly_kpis["week_start"] = pd.to_datetime(weekly_kpis["week_start"])
for c in ["impressions", "spend", "vv", "conversions", "order_value", "vast_start", "vast_complete"]:
    weekly_kpis[c] = pd.to_numeric(weekly_kpis[c], errors="coerce").astype(float)
weekly_kpis = weekly_kpis[weekly_kpis["impressions"] >= MIN_WEEKLY_IMPRESSIONS].copy()

print(f"{len(weekly_kpis):,} records | {weekly_kpis['advertiser_id'].nunique()} advertisers")
print(f"Date range: {weekly_kpis['week_start'].min().date()} to {weekly_kpis['week_start'].max().date()}")

## 4. Compute Metrics & Platform Covariates

In [ ]:
def compute_metrics(df):
    df = df.copy()
    df["ivr"] = df["vv"] / df["impressions"].replace(0, np.nan)
    df["cvr"] = df["conversions"] / df["vv"].replace(0, np.nan)
    df["cpa"] = df["spend"] / df["conversions"].replace(0, np.nan)
    df["cpv"] = df["spend"] / df["vv"].replace(0, np.nan)
    df["roas"] = df["order_value"] / df["spend"].replace(0, np.nan)
    df["vcr"] = df["vast_complete"] / df["vast_start"].replace(0, np.nan)
    return df

def winsorize(s, lo=1, hi=99):
    return s.clip(lower=np.nanpercentile(s.dropna(), lo), upper=np.nanpercentile(s.dropna(), hi))

# Aggregate to advertiser-week
adv_week = weekly_kpis.groupby(["advertiser_id", "week_start", "is_adopter"]).agg(
    impressions=("impressions", "sum"), spend=("spend", "sum"), vv=("vv", "sum"),
    conversions=("conversions", "sum"), order_value=("order_value", "sum"),
    vast_start=("vast_start", "sum"), vast_complete=("vast_complete", "sum"),
).reset_index()

# Platform covariates from non-adopters
non_adopter = adv_week[~adv_week["is_adopter"]]
platform = non_adopter.groupby("week_start").agg(
    platform_impressions=("impressions", "sum"), platform_spend=("spend", "sum"),
    platform_vv=("vv", "sum"), platform_vast_start=("vast_start", "sum"),
    platform_vast_complete=("vast_complete", "sum"),
    platform_active_advertisers=("advertiser_id", "nunique"),
).reset_index()
platform["platform_ivr"] = platform["platform_vv"] / platform["platform_impressions"].replace(0, np.nan)
platform["platform_vcr"] = platform["platform_vast_complete"] / platform["platform_vast_start"].replace(0, np.nan)
platform["holiday"] = platform["week_start"].map(lambda w: HOLIDAY_WEEKS.get(w, 0.0))
platform["platform_spend"] /= 1e6
platform["platform_impressions"] /= 1e9
platform["platform_active_advertisers"] /= 1000.0

print(f"Platform covariates: {len(platform)} weeks from {non_adopter['advertiser_id'].nunique()} non-adopter advertisers")
platform.tail()

## 5. Run CausalImpact Analysis

For each adopter:
1. Get their weekly prospecting KPIs (ALL campaigns, advertiser-level)
2. Split at their intervention date (first recommended plan)
3. Run CausalImpact with platform covariates + lagged metric

In [ ]:
def run_ci(adv_id, intervention_date, metric, adv_week, platform):
    """Run CausalImpact for one advertiser. Returns result dict or None."""
    data = adv_week[adv_week["advertiser_id"] == adv_id].copy()
    data = compute_metrics(data)
    for m in ["ivr", "cvr", "cpa", "cpv", "roas"]:
        if m in data.columns:
            data[m] = winsorize(data[m])
    data = data.merge(platform, on="week_start", how="inner")
    data = data.sort_values("week_start")
    data["metric_lag1"] = data[metric].shift(1)
    data = data.dropna(subset=["metric_lag1"]).set_index("week_start").sort_index()

    int_ts = pd.Timestamp(intervention_date)
    int_week = int_ts - pd.Timedelta(days=int_ts.weekday())
    pre, post = data[data.index < int_week], data[data.index >= int_week]

    if len(pre) < MIN_PRE_WEEKS or len(post) < MIN_POST_WEEKS:
        return None
    if post["spend"].sum() < MIN_TOTAL_POST_SPEND:
        return None

    pre_period = [pre.index[0], pre.index[-1]]
    post_period = [post.index[0], post.index[-1]]

    covs = [c for c in COVARIATES + ["metric_lag1"] if c in data.columns]
    ci_data = data[[metric] + covs].dropna(subset=[metric]).astype(float)
    ci_data[covs] = ci_data[covs].ffill().bfill()

    if len(ci_data) < 10:
        return None

    try:
        ci = CausalImpact(ci_data, pre_period, post_period)
        inf = ci.inferences[ci.inferences.index >= post_period[0]]
        predicted = inf["preds"].mean()
        abs_eff = inf["point_effects"].mean()
        return {
            "advertiser_id": adv_id,
            "pre_weeks": len(pre), "post_weeks": len(post),
            "actual_avg": ci.post_data.iloc[:, 0].mean(),
            "predicted_avg": predicted,
            "absolute_effect": abs_eff,
            "relative_effect": abs_eff / predicted if predicted != 0 else np.nan,
            "ci_lower": inf["point_effects_lower"].mean(),
            "ci_upper": inf["point_effects_upper"].mean(),
            "p_value": ci.p_value,
            "significant": ci.p_value < 0.05,
            "post_spend": float(post["spend"].sum()),
            "ci_object": ci,
        }
    except Exception as e:
        print(f"  Error: {e}")
        return None

In [ ]:
results = []
for _, row in adopters.iterrows():
    r = run_ci(row["advertiser_id"], row["intervention_date"], PRIMARY_METRIC, adv_week, platform)
    if r:
        r["company_name"] = row["company_name"]
        results.append(r)
        sig = "*" if r["significant"] else ""
        print(f"{row['company_name']:35s} ({row['advertiser_id']}): {r['relative_effect']:+.2%} "
              f"(p={r['p_value']:.4f}) {sig}  [{r['pre_weeks']}w pre, {r['post_weeks']}w post]")
    else:
        print(f"{row['company_name']:35s} ({row['advertiser_id']}): skipped")

print(f"\n{len(results)} advertisers analyzed")

## 6. Results Summary

In [ ]:
def summarize(results, metric):
    df = pd.DataFrame([{k: v for k, v in r.items() if k != "ci_object"} for r in results])
    df["effect_pct"] = df["relative_effect"].apply(lambda x: f"{x:+.2%}")
    df["p_fmt"] = df["p_value"].apply(lambda x: f"{x:.4f}")
    df["sig"] = df["significant"].apply(lambda x: "Yes" if x else "")

    display(df[["company_name", "advertiser_id", "pre_weeks", "post_weeks",
               "actual_avg", "predicted_avg", "effect_pct", "p_fmt", "sig"]]
           .style.set_caption(f"Causal Impact — {METRIC_DEFS[metric]['label']}"))

    d = METRIC_DEFS[metric]["direction"]
    n = len(df)
    n_sig = df["significant"].sum()
    n_pos = (df["relative_effect"] > 0).sum() if d == "higher" else (df["relative_effect"] < 0).sum()
    ts = df["post_spend"].sum()
    wt = (df["relative_effect"] * df["post_spend"]).sum() / ts if ts > 0 else 0

    print(f"\nAdvertisers analyzed:       {n}")
    print(f"Statistically significant:  {n_sig} ({n_sig/n:.0%})")
    print(f"Positive outcome:           {n_pos} ({n_pos/n:.0%})")
    print(f"Mean effect:                {df['relative_effect'].mean():+.2%}")
    print(f"Median effect:              {df['relative_effect'].median():+.2%}")
    print(f"Spend-weighted effect:      {wt:+.2%}")
    return df

summary_ivr = summarize(results, PRIMARY_METRIC)

## 7. Per-Advertiser CausalImpact Plots

In [ ]:
for r in results:
    print(f"\n{'='*80}")
    print(f"{r['company_name']} ({r['advertiser_id']}) — {PRIMARY_METRIC.upper()}")
    print(f"Effect: {r['relative_effect']:+.2%} (p={r['p_value']:.4f})")
    print(f"{'='*80}")
    r["ci_object"].plot(figsize=(14, 10))
    plt.show()

## 8. Aggregate Summary Chart

In [ ]:
df_plot = summary_ivr.sort_values("relative_effect")
d = METRIC_DEFS[PRIMARY_METRIC]["direction"]
colors = ["#2ecc71" if (s and ((d=="higher" and e>0) or (d=="lower" and e<0)))
          else "#e74c3c" if s else "#95a5a6"
          for s, e in zip(df_plot["significant"], df_plot["relative_effect"])]

fig, ax = plt.subplots(figsize=(12, max(6, len(df_plot) * 0.6)))
bars = ax.barh(df_plot["company_name"], df_plot["relative_effect"] * 100, color=colors)
ax.axvline(x=0, color="black", linewidth=0.8)
for bar, pval in zip(bars, df_plot["p_value"]):
    x = bar.get_width()
    ax.text(x + (1 if x >= 0 else -1), bar.get_y() + bar.get_height() / 2,
            f"p={pval:.3f}", va="center", ha="left" if x >= 0 else "right", fontsize=9)
ax.set_xlabel("Relative Effect (%)")
ax.set_title(f"Media Plan Causal Impact — {PRIMARY_METRIC.upper()} (Recommended Plans Only)\n"
             "Green = significant positive | Red = significant negative | Gray = not significant")
plt.tight_layout()
plt.show()

## 9. Within-Advertiser Comparison

For advertisers with BOTH recommended and non-recommended campaigns running in the post-period, compare their performance side-by-side.

**Important caveat:** Recommended campaigns are *new* campaigns (created with media plan), while non-recommended campaigns are typically *older* campaigns with established audience patterns. New campaigns generally underperform mature ones during their ramp-up phase, so a lower IVR for recommended campaigns does not necessarily mean the recommendation is bad — it may reflect campaign maturity differences.

In [ ]:
# Load recommended campaign group IDs
rec_cgs = set(client.query("""
SELECT mp.campaign_group_id
FROM `dw-main-silver.core.media_plan` mp
JOIN `dw-main-silver.core.media_plan_publishers` mpp ON mpp.media_plan_id = mp.media_plan_id
WHERE mp.media_plan_status_id = 3
GROUP BY 1
HAVING LOGICAL_AND(mpp.badge_state = 'RECOMMENDED')
""").to_dataframe()["campaign_group_id"].tolist())

comparison = []
for _, row in adopters.iterrows():
    adv_id = row["advertiser_id"]
    int_week = pd.Timestamp(row["intervention_date"]) - pd.Timedelta(days=pd.Timestamp(row["intervention_date"]).weekday())
    post = weekly_kpis[(weekly_kpis["advertiser_id"] == adv_id) & (weekly_kpis["week_start"] >= int_week)]
    rec = post[post["campaign_group_id"].isin(rec_cgs)]
    non = post[~post["campaign_group_id"].isin(rec_cgs)]
    if rec.empty: continue
    r_agg = rec[["impressions","spend","vv","conversions","order_value"]].sum()
    n_agg = non[["impressions","spend","vv","conversions","order_value"]].sum() if not non.empty else pd.Series({"impressions":0,"spend":0,"vv":0,"conversions":0,"order_value":0})
    comparison.append({
        "company_name": row["company_name"], "advertiser_id": adv_id,
        "has_non_rec": not non.empty,
        "rec_ivr": r_agg["vv"]/r_agg["impressions"] if r_agg["impressions"]>0 else np.nan,
        "non_rec_ivr": n_agg["vv"]/n_agg["impressions"] if n_agg["impressions"]>0 else np.nan,
        "rec_cvr": r_agg["conversions"]/r_agg["vv"] if r_agg["vv"]>0 else np.nan,
        "non_rec_cvr": n_agg["conversions"]/n_agg["vv"] if n_agg["vv"]>0 else np.nan,
        "rec_spend": r_agg["spend"], "non_rec_spend": n_agg["spend"],
    })

comp_df = pd.DataFrame(comparison)
display(comp_df.style.set_caption("Within-Advertiser: Recommended vs Non-Recommended (Post-Period)").format(precision=4))

both = comp_df[comp_df["has_non_rec"]]
if not both.empty:
    print(f"\nAdvertisers with both types: {len(both)}")
    print(f"Avg IVR diff (rec - non_rec): {(both['rec_ivr'] - both['non_rec_ivr']).mean():+.4f}")

## 10. All Metrics (Optional)

In [ ]:
all_summaries = {}
for metric in METRIC_DEFS:
    print(f"\n{'='*60}\n{metric.upper()} ({METRIC_DEFS[metric]['label']})\n{'='*60}")
    mr = []
    for _, row in adopters.iterrows():
        r = run_ci(row["advertiser_id"], row["intervention_date"], metric, adv_week, platform)
        if r:
            r["company_name"] = row["company_name"]
            mr.append(r)
    if mr:
        all_summaries[metric] = summarize(mr, metric)

## 11. Cross-Metric Summary

In [ ]:
cross = []
for metric, df in all_summaries.items():
    info = METRIC_DEFS[metric]
    n = len(df); n_sig = df["significant"].sum()
    n_pos = (df["relative_effect"] > 0).sum() if info["direction"] == "higher" else (df["relative_effect"] < 0).sum()
    ts = df["post_spend"].sum()
    wt = (df["relative_effect"] * df["post_spend"]).sum() / ts if ts > 0 else 0
    cross.append({"Metric": metric.upper(), "Label": info["label"], "N": n,
                  "Significant": f"{n_sig}/{n}", "Positive": f"{n_pos}/{n}",
                  "Mean": f"{df['relative_effect'].mean():+.2%}",
                  "Median": f"{df['relative_effect'].median():+.2%}",
                  "Spend-Weighted": f"{wt:+.2%}"})
display(pd.DataFrame(cross).style.set_caption("Cross-Metric Summary"))

---

## Appendix A: How CausalImpact Works

### The Core Idea

We can't directly observe what *would have happened* to an advertiser's performance if they hadn't adopted media plan (the **counterfactual**). CausalImpact solves this by building a statistical model that learns the advertiser's behavior patterns from the pre-period, then uses that model to predict what *should have happened* in the post-period.

The difference between the **actual** post-period performance and the **predicted** (counterfactual) performance is the estimated causal effect.

### Step-by-Step Process

```
PRE-PERIOD (52 weeks before adoption):
┌─────────────────────────────────────────────────────────────┐
│ The model learns:                                           │
│   IVR = f(platform_IVR, platform_spend, holidays, lag, ...) │
│                                                             │
│ "When the platform IVR is X, and it's holiday season,       │
│  this advertiser's IVR tends to be Y"                       │
└─────────────────────────────────────────────────────────────┘
                              │
                              ▼
POST-PERIOD (after adoption):
┌─────────────────────────────────────────────────────────────┐
│ The model predicts: "Given what the platform did in the     │
│ post-period, this advertiser's IVR SHOULD HAVE been Z       │
│ (if nothing changed)"                                       │
│                                                             │
│ Actual IVR = Z'                                             │
│ Causal Effect = Z' - Z                                      │
│ Relative Effect = (Z' - Z) / Z                              │
└─────────────────────────────────────────────────────────────┘
```

### The Statistical Model (Bayesian Structural Time Series)

Under the hood, CausalImpact uses a **Bayesian structural time series** model with three components:

1. **Local linear trend:** Captures the advertiser's own trajectory (is IVR trending up or down over time?)
2. **Regression on covariates:** Learns how the advertiser's metric relates to external factors (platform IVR, holidays, etc.)
3. **Bayesian inference:** Instead of a single prediction, the model generates a **probability distribution** of possible counterfactuals, giving us uncertainty bounds.

### How P-Values Are Calculated

CausalImpact computes p-values using a **Bayesian tail-area probability:**

1. The model generates thousands of simulated counterfactual paths from the posterior distribution
2. For each simulation, it calculates the cumulative effect (sum of pointwise differences)
3. The p-value = the fraction of simulated effects that are more extreme than zero in the same direction as the observed effect
4. If p = 0.02, it means only 2% of the simulated counterfactuals showed an effect as large as what we observed — strong evidence that the intervention had a real impact

**Note:** These are *one-sided* p-values. p < 0.05 means there's < 5% chance the observed effect occurred by chance.

### Why Covariates Matter

Without covariates, the model only has the advertiser's own history to predict the counterfactual. If something external changed (like Q4 holiday season boosting everyone's metrics), the model might incorrectly attribute that to the media plan.

Our covariates solve this:

| Covariate | What It Controls For |
|---|---|
| `platform_ivr` | Market-wide engagement trends (if everyone's IVR drops, it's not media plan's fault) |
| `platform_spend` | Industry spending patterns / seasonality |
| `platform_impressions` | Supply-side changes (more/fewer impressions available) |
| `holiday` | Known seasonal spikes (Black Friday, Christmas, Super Bowl) |
| `platform_active_advertisers` | Competition changes (more advertisers = harder to win good placements) |
| `platform_vcr` | Video completion rate — CTV engagement proxy not affected by network allocation |
| `metric_lag1` | The advertiser's own previous-week value — captures momentum/autocorrelation |

### Key Assumptions

1. **Parallel trends:** Without the intervention, the advertiser would have continued following the pattern predicted by covariates
2. **No spillover:** Media plan adoption doesn't affect non-adopter performance (our covariates)
3. **Stable relationships:** The covariate relationships learned in pre-period remain valid in post-period

### Limitations

1. **Selection bias:** Advertisers who chose to adopt may be systematically different (more engaged, growing faster)
2. **Confounders:** Other changes (new creatives, budget shifts, audience changes) happening at the same time
3. **Small N:** With only ~7 analyzable advertisers, individual results are noisy
4. **Campaign maturity:** New media plan campaigns are compared to established ones — maturity effects may mask true impact